In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy
import json
import numpy as np

In [ ]:
annot_test = pd.read_csv("/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_test.csv")

In [ ]:
annot_train = pd.read_csv("/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_train.csv")
annot_train.head()

In [ ]:
annot_train[annot_train.image_id == '000434271f63a053c4128a0ba6352c7f']

In [ ]:
annot_train

In [ ]:
annot_train[annot_train.image_id == "0005e8e3701dfb1dd93d53e2ff537b6e"]

In [ ]:
annot_train[annot_train.class_name == "Pleural thickening"]


In [ ]:
annot_test.head()

In [ ]:
annot_test.shape

In [ ]:
annot_test.image_id.value_counts()

In [ ]:
annot_test[annot_test.class_name == "Edema"]

In [ ]:
annot_test[annot_test.image_id == "ab9dedb9ff4cd9e80dca74505b599105"].class_name.unique()

In [ ]:
img_lbl_test = pd.read_csv("/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/image_labels_test.csv")

In [ ]:
img_lbl_test.head().T

In [ ]:
img_lbl_test.head()

In [ ]:
img_lbl_test.Edema.value_counts()

In [ ]:
img_lbl_test.columns[1:]

In [ ]:
set(img_lbl_test.columns[1:]) - {'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
 'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other', 'Pneumonia', 'Pneumothorax', 'Support Devices'}

In [ ]:
images_root = "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr"
train_meta = pd.read_csv(Path(images_root) / "train_meta.csv")

In [ ]:
train_meta.head()

In [ ]:
test_meta = pd.read_csv(Path(images_root) / "test_meta.csv")
test_meta.head()

In [ ]:
x, y = train_meta.loc[train_meta['image_id'] == '4d390e07733ba06e5ff07412f09c0a92', 'dim0'].values, train_meta.loc[train_meta['image_id'] == '4d390e07733ba06e5ff07412f09c0a92', 'dim1'].values

In [ ]:
x, y

In [ ]:
def load_meta(meta_csv):
    """
    Load VinDr meta CSV with columns: image_id, dim0, dim1
    Returns dict: image_id -> (orig_height, orig_width)
    """
    df = pd.read_csv(meta_csv)
    meta = {}
    for _, r in df.iterrows():
        image_id = r["image_id"]
        # dim0 = height, dim1 = width (typical numpy convention)
        orig_h = int(r["dim0"])
        orig_w = int(r["dim1"])
        meta[image_id] = (orig_h, orig_w)
    return meta

In [ ]:
def visualize_vindr_boxes(
    image_id: str,
    images_root: str,
    annotations_csv: str,
    meta_csv: str,
    save_path: str = None,
):
    """
    Visualize a 256x256 VinDr PNG with its bounding boxes,
    scaling the original annotations using train_meta.csv.
    """
    # Load annotations
    df = pd.read_csv(annotations_csv)
    rows = df[df["image_id"] == image_id]
    if rows.empty:
        raise ValueError(f"No annotations found for image_id={image_id} in {annotations_csv}")

    # Load meta for original size
    meta = load_meta(meta_csv)
    if image_id not in meta:
        raise KeyError(f"image_id={image_id} not found in meta CSV {meta_csv}")
    orig_h, orig_w = meta[image_id]

    # Load 256x256 PNG
    img_path = Path(images_root) / f"{image_id}.png"
    if not img_path.exists():
        raise FileNotFoundError(f"PNG not found: {img_path}")
    img = Image.open(img_path).convert("RGB")
    w_png, h_png = img.size

    assert w_png == h_png == 256, f"Expected 256x256 PNG, got {w_png}x{h_png}"

    # Scaling factors from original -> 256x256
    scale_x = w_png / float(orig_w)
    scale_y = h_png / float(orig_h)

    fig, ax = plt.subplots(1, figsize=(6, 6))
    ax.imshow(img, cmap="gray")
    ax.axis("off")

    for _, r in rows.iterrows():
        x_min_orig = r["x_min"]
        y_min_orig = r["y_min"]
        x_max_orig = r["x_max"]
        y_max_orig = r["y_max"]
        label = r.get("class_name", "")

        # Scale to PNG coordinates
        x_min = x_min_orig * scale_x
        y_min = y_min_orig * scale_y
        x_max = x_max_orig * scale_x
        y_max = y_max_orig * scale_y

        width = x_max - x_min
        height = y_max - y_min

        rect = patches.Rectangle(
            (x_min, y_min),
            width,
            height,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
        )
        ax.add_patch(rect)
        if label:
            ax.text(
                x_min,
                max(y_min - 2, 0),
                label,
                fontsize=8,
                color="yellow",
                bbox=dict(facecolor="black", alpha=0.5, edgecolor="none"),
            )

    plt.tight_layout()
    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150)
        print(f"Saved visualization to {save_path}")
    else:
        plt.show()

In [ ]:
visualize_vindr_boxes(
    "0005e8e3701dfb1dd93d53e2ff537b6e",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_train.csv",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train_meta.csv",
)

In [ ]:
visualize_vindr_boxes(
    "0007d316f756b3fa0baea2ff514ce945",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_train.csv",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/train_meta.csv",
)

In [ ]:
img_lbl_test.columns[1:].values

In [ ]:
with open("/home/harsh/Documents/fau/thesis/thesis-codebase/ALBEF/vindr_zero_shot_results_a40/vindr_zero_shot_all_checkpoints.json", "r") as f:
    zsclf = json.load(f)

In [ ]:
pd.DataFrame(zsclf)

In [ ]:
# -------------------------
# 1. SUMMARY TABLE
# -------------------------
summary_rows = []
ckpt_rename = {
    "checkpoint_09.pth": "ckpt 10",
    "checkpoint_19.pth": "ckpt 20",
    "checkpoint_29.pth": "ckpt 30",
    "checkpoint_best.pth": "ckpt best avg loss"
}

for ckpt_name, ckpt_data in zsclf.items():
    cls = ckpt_data["classification"]

    summary_rows.append({
        "checkpoint": ckpt_rename[ckpt_name],
        "macro_auc": cls["macro_auc"],
        "micro_auc": cls["micro_auc"],
        "map_at_10": cls["map_at_10"],
    })

df_summary = pd.DataFrame(summary_rows)
df_summary = df_summary.set_index("checkpoint")
df_summary

In [ ]:
df_summary.to_csv("../results/zero_shot_clf_vindr/auc_summary_24_11_2025.csv")

In [ ]:
# -------------------------
# 2. PER-LABEL AUC TABLE
# -------------------------

per_label_dict = {}

for ckpt_name, ckpt_data in zsclf.items():
    per_auc = ckpt_data["classification"]["per_label_auc"]

    # Convert None → NaN for readability
    row = {label: (auc if auc is not None else float("nan"))
           for label, auc in per_auc.items()}

    per_label_dict[ckpt_name] = row

# Build DataFrame: rows = labels, columns = checkpoints
df_per_label = pd.DataFrame(per_label_dict)
df_per_label

In [ ]:
df_per_label.rename({
    "checkpoint_09.pth": "ckpt 10",
    "checkpoint_19.pth": "ckpt 20",
    "checkpoint_29.pth": "ckpt 30",
    "checkpoint_best.pth": "ckpt best avg loss"
}, axis="columns", inplace=True)

In [ ]:
df_per_label

In [ ]:
df_per_label.to_csv("../results/zero_shot_clf_vindr/per_label_auc_24_11_2025.csv")

In [ ]:
def scale_boxes_to_256(ann_df, meta_df):
    """
    Join annotations with image dimensions and rescale boxes
    from (width, height) -> (256, 256).
    """
    print("Merging annotations with meta information (image dimensions).")

    merged = ann_df.merge(meta_df, on="image_id", how="left", validate="many_to_one")

    if merged["dim1"].isna().any() or merged["dim0"].isna().any():
        missing_ids = merged.loc[
            merged["dim1"].isna() | merged["dim0"].isna(), "image_id"
        ].unique()
        raise ValueError(
            f"Missing width/height for some image_ids in test_meta.csv. "
            f"Examples: {missing_ids[:10]}"
        )

    # Avoid division by zero
    if (merged["dim1"] <= 0).any() or (merged["dim0"] <= 0).any():
        bad = merged[(merged["dim1"] <= 0) | (merged["dim0"] <= 0)]
        raise ValueError(
            f"Found non-positive width/height for some rows, e.g.:\n{bad.head()}"
        )

    target_size = 256.0

    print("Scaling bounding boxes to 256x256 coordinate space.")
    merged["scale_x"] = target_size / merged["dim1"]
    merged["scale_y"] = target_size / merged["dim0"]

    for coord in ["x_min", "x_max"]:
        merged[f"{coord}_256"] = merged[coord] * merged["scale_x"]

    for coord in ["y_min", "y_max"]:
        merged[f"{coord}_256"] = merged[coord] * merged["scale_y"]

    # Clip to [0, 256] (just to be safe)
    for coord in ["x_min_256", "x_max_256", "y_min_256", "y_max_256"]:
        merged[coord] = merged[coord].clip(0.0, target_size)

    # Ensure min <= max after potential rounding/clipping
    merged["x_min_256"], merged["x_max_256"] = np.minimum(
        merged["x_min_256"], merged["x_max_256"]
    ), np.maximum(merged["x_min_256"], merged["x_max_256"])
    merged["y_min_256"], merged["y_max_256"] = np.minimum(
        merged["y_min_256"], merged["y_max_256"]
    ), np.maximum(merged["y_min_256"], merged["y_max_256"])

    out_cols = [
        "image_id",
        "class_name",
        "x_min_256",
        "y_min_256",
        "x_max_256",
        "y_max_256",
    ]

    out_df = merged[out_cols].copy()
    out_df = out_df.rename(
        columns={
            "class_name": "label_name",
            "x_min_256": "x_min",
            "y_min_256": "y_min",
            "x_max_256": "x_max",
            "y_max_256": "y_max",
        }
    )

    print(f"Total GT boxes: {len(out_df)}")
    print(f"Unique images in GT: {out_df['image_id'].nunique()}")
    return out_df

In [ ]:
df = scale_boxes_to_256(annot_test, test_meta)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
annot_test.head()

In [ ]:
visualize_vindr_boxes(
    "e0dc2e79105ad93532484e956ef8a71a",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/test",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_test.csv",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/test_meta.csv",
)

In [ ]:
visualize_vindr_boxes(
    "e0dc2e79105ad93532484e956ef8a71a",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/test",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/annotations/annotations_test.csv",
    "/home/harsh/Documents/fau/thesis/thesis-codebase/data/vindr_cxr/test_meta.csv",
)